# 🧠 Notebook 06 — LangChain RAG Pipeline
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Build a RAG pipeline: embed query → retrieve top-5 → generate answer
- Use `flan-t5-base` as the free local LLM
- Append mandatory medical disclaimer to every response
- Test on 10 sample queries with full logging
- Build and verify `src/rag/pipeline.py`

### 📋 Deliverables
- `notebooks/05_rag_pipeline.ipynb`
- `src/rag/pipeline.py`

---

## 1. Imports & Setup

In [1]:
import os
import sys
import time
import json
import pandas as pd

# Add project root to path
sys.path.append(os.path.abspath('..'))

print("✅ Imports ready")

✅ Imports ready


## 2. Build RAG Pipeline

This imports from `src/rag/pipeline.py` which implements:
1. **Encoder**: `all-MiniLM-L6-v2` (sentence-transformers)
2. **Vector store**: FAISS `IndexFlatL2`
3. **Generator**: `google/flan-t5-base` (free, local, no API key)
4. **Disclaimer**: Appended to every response automatically

In [2]:
from src.rag.pipeline import build_rag_pipeline

pipeline = build_rag_pipeline()

D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading FAISS index: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\data\embeddings\faiss_index\pubmedqa_index_flatl2.faiss
  Vectors in index: 9,800
Loading chunk mapping: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\data\embeddings\faiss_index\chunk_mapping.pkl
✅ BM25 index built over 9,800 documents
GROQ_API_KEY not set — falling back to local flan-t5-base
✅ RAG Pipeline ready


## 3. Test Retrieval (Top-5)

Verify that FAISS returns relevant medical context for a sample query.

In [3]:
test_query = "What are the symptoms of diabetes?"

retrieved = pipeline.retrieve(test_query, top_k=5)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")

for i, chunk in enumerate(retrieved):
    print(f"--- Chunk {i+1} | ID: {chunk['chunk_id']} | "
          f"Category: {chunk['category']} | Distance: {chunk['distance']:.4f} ---")
    print(f"Q: {chunk['question'][:100]}")
    print(f"A: {chunk['answer'][:150]}")
    print()

Query: What are the symptoms of diabetes?
Retrieved 5 chunks:

--- Chunk 1 | ID: 294 | Category: Symptoms | Distance: -15.2510 ---
Q: Do traditional symptoms of hypothyroidism correlate with biochemical disease?
A: In this sample, the number of hypothyroid symptoms reported was directly related to the level of TSH. The association was stronger when more symptoms 

--- Chunk 2 | ID: 5728 | Category: Prevention | Distance: -14.7089 ---
Q: Is prehypertension a risk factor for the development of type 2 diabetes?
A: Subjects with prehypertension are at increased risk of diabetes. Much of this risk is explained by disorders related to the insulin resistance syndrom

--- Chunk 3 | ID: 2887 | Category: Diagnosis | Distance: -14.5955 ---
Q: Are zinc transporter type 8 antibodies a marker of autoimmune thyroiditis in non-obese adults with n
A: In non-obese adults with new-onset diabetes, the presence of GADA and especially ZnT8 autoantibodies increases the risk of AITD.

--- Chunk 4 | ID: 8727 |

## 4. Test Full Pipeline (Single Query)

Full chain: embed → retrieve top-5 → build prompt → generate → add disclaimer.

In [4]:
result = pipeline.answer(test_query)

print(f"Question: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nDisclaimer present: {result['disclaimer_present']}")
print(f"Sources retrieved: {result['top_k']}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1205 > 512). Running this sequence through the model will result in indexing errors


Question: What are the symptoms of diabetes?

Answer:
GADA, ICA, IA-2A and ZnT8A.

⚠️ MEDICAL DISCLAIMER: This response is generated by an AI system for educational and informational purposes only. It is NOT a substitute for professional medical advice, diagnosis, or treatment. Always consult a qualified healthcare provider for medical decisions.

Disclaimer present: True
Sources retrieved: 5


## 5. Test on 10 Sample Queries (with Logging)

The plan requires testing on 10 queries with logged context + generated answer.

In [5]:
test_queries = [
    # Symptoms
    "What are the early symptoms of type 2 diabetes?",
    # Diagnosis
    "How is pneumonia diagnosed in elderly patients?",
    # Treatment
    "What are the current treatment options for hypertension?",
    # Medication
    "What are the side effects of metformin?",
    # Prevention
    "How can cardiovascular disease be prevented through lifestyle changes?",
    # General
    "What is the role of the immune system in fighting infections?",
    # Mixed / clinical
    "Is laparoscopic surgery better than open surgery for prostatectomy?",
    "Can bacterial gastroenteritis lead to irritable bowel syndrome?",
    "Does naturopathy effectively treat menopausal symptoms?",
    "Is urgent colonoscopy needed for acute diverticular bleeding?",
]


In [6]:
results = []
latencies = []

print("=" * 100)
print("RUNNING 10-QUERY TEST")
print("=" * 100)

for i, query in enumerate(test_queries, 1):
    start = time.time()
    result = pipeline.answer(query)
    elapsed = (time.time() - start) * 1000
    latencies.append(elapsed)

    results.append({
        "query_num": i,
        "question": result["question"],
        "answer_raw": result["answer_raw"],
        "disclaimer_present": result["disclaimer_present"],
        "top_k": result["top_k"],
        "sources": result["retrieved_sources"],
        "latency_ms": round(elapsed, 2),
    })

    print(f"\n{'─' * 100}")
    print(f"Query {i}: {query}")
    print(f"⏱️  Latency: {elapsed:.0f}ms")
    print(f"📂 Sources: {[s['chunk_id'] for s in result['retrieved_sources']]}")
    print(f"📝 Answer: {result['answer_raw'][:200]}")
    print(f"⚠️  Disclaimer: {result['disclaimer_present']}")

RUNNING 10-QUERY TEST

────────────────────────────────────────────────────────────────────────────────────────────────────
Query 1: What are the early symptoms of type 2 diabetes?
⏱️  Latency: 2873ms
📂 Sources: [40, 5728, 4555, 7064, 4249]
📝 Answer: Incident impaired fasting glucose (IFG), impaired glucose tolerance (IGT), and newly diagnosed type 2 diabetes were associated with increased bodily pain at baseline compared with those with normal gl
⚠️  Disclaimer: True

────────────────────────────────────────────────────────────────────────────────────────────────────
Query 2: How is pneumonia diagnosed in elderly patients?
⏱️  Latency: 1512ms
📂 Sources: [5415, 3532, 9721, 501, 8563]
📝 Answer: Pneumonia in elderly patients is diagnosed by bronchitis.
⚠️  Disclaimer: True

────────────────────────────────────────────────────────────────────────────────────────────────────
Query 3: What are the current treatment options for hypertension?
⏱️  Latency: 1446ms
📂 Sources: [8895, 7608, 4494, 

## 6. Results Summary

In [7]:
print("\n" + "=" * 60)
print("📊 10-QUERY TEST SUMMARY")
print("=" * 60)

print(f"\nQueries tested:    {len(results)}")
print(f"All with disclaimer: {all(r['disclaimer_present'] for r in results)}")
print(f"All with 5 sources:  {all(r['top_k'] == 5 for r in results)}")

print(f"\n⏱️  Latency:")
print(f"  Min:    {min(latencies):.0f}ms")
print(f"  Max:    {max(latencies):.0f}ms")
print(f"  Mean:   {sum(latencies)/len(latencies):.0f}ms")

# Check disclaimer in every response
for r in results:
    assert r["disclaimer_present"], f"Missing disclaimer for query: {r['question']}"
print("\n✅ All 10 queries passed — disclaimer present, 5 sources retrieved")


📊 10-QUERY TEST SUMMARY

Queries tested:    10
All with disclaimer: True
All with 5 sources:  True

⏱️  Latency:
  Min:    1446ms
  Max:    4445ms
  Mean:   2633ms

✅ All 10 queries passed — disclaimer present, 5 sources retrieved


## 7. Save Test Results Log

In [8]:
log_path = "../reports/rag_pipeline_test_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Test log saved to: {log_path}")

✅ Test log saved to: ../reports/rag_pipeline_test_log.json


## 8. Verify src/rag/pipeline.py Module

Confirm the module is importable and functional independently.

In [9]:
# Test that module-level convenience functions work
from src.rag.pipeline import answer as rag_answer

quick_result = rag_answer("What causes fever in children?")
print(f"Module-level answer() works: {len(quick_result['answer']) > 0}")
print(f"Answer preview: {quick_result['answer_raw'][:150]}")

Module-level answer() works: True
Answer preview: Pneumonia is an uncommon cause of fever and neutropenia in children with cancer.


## ✅ Summary

| Item | Status |
|---|---|
| LLM | `google/flan-t5-base` (free, local) |
| Retrieval | FAISS top-5 |
| Disclaimer | Appended to every response |
| Test queries | 10 queries logged with context + answer |
| `src/rag/pipeline.py` | Built and verified |

---

### ➡️ Next Step
Open **`06_classification_model.ipynb`** to fine-tune DistilBERT classifier.